# Assignment 2

**COMP3361 Natural Language Processing**

**Student Name:** Pang Tin Hei  
**University Number:** 3036100179

In [1]:
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score
from seqeval.metrics import f1_score as seq_f1_score
from datasets import load_dataset
from tqdm import tqdm
import os

### Import Data

In [ ]:
# Configurations
DATA_DIR = "./ontonotes5/dataset"
BATCH_SIZE = 32
MAX_LEN = 128
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load labels mapping
with open(os.path.join(DATA_DIR, "label.json"), "r") as f:
    label2id = json.load(f)
id2label = {int(v): k for k, v in label2id.items()}
num_labels = len(label2id)


def load_data(file_paths):
    data = []
    for fp in file_paths:
        with open(os.path.join(DATA_DIR, fp), "r") as f:
            for line in f:
                data.append(json.loads(line))
    return data


train_files = ["train00.json", "train01.json", "train02.json", "train03.json"]
train_data = load_data(train_files)
valid_data = load_data(["valid.json"])
test_data = load_data(["test.json"])

# Build Vocabulary
vocab = {"<PAD>": 0, "<UNK>": 1}
for ex in train_data:
    for word in ex["tokens"]:
        if word not in vocab:
            vocab[word] = len(vocab)


class NERDataset(Dataset):
    def __init__(self, data, vocab, max_len):
        self.data = data
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        tokens = [self.vocab.get(w, self.vocab["<UNK>"]) for w in item["tokens"]]
        tags = [int(t) for t in item["tags"]]

        # Pading/Truncating
        if len(tokens) > self.max_len:
            tokens = tokens[: self.max_len]
            tags = tags[: self.max_len]
        else:
            pad_len = self.max_len - len(tokens)
            tokens.extend([self.vocab["<PAD>"]] * pad_len)
            tags.extend([-100] * pad_len)

        return torch.tensor(tokens), torch.tensor(tags)


train_dataset = NERDataset(train_data, vocab, MAX_LEN)
valid_dataset = NERDataset(valid_data, vocab, MAX_LEN)
test_dataset = NERDataset(test_data, vocab, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

Using device: cuda


### BiLSTM Tagger

In [3]:
class BiLSTMTagger(nn.Module):
    def __init__(self, vocab_size, label_size, embed_dim, hidden_dim):
        super(BiLSTMTagger, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim // 2,
            num_layers=2,
            bidirectional=True,
            batch_first=True,
            dropout=0.3,
        )
        self.fc = nn.Linear(hidden_dim, label_size)

    def forward(self, input_ids):
        embedded = self.embedding(input_ids)
        lstm_out, _ = self.lstm(embedded)
        emissions = self.fc(lstm_out)
        return emissions


EMBED_DIM = 64
HIDDEN_DIM = 128
EPOCHS = 3
LEARNING_RATE = 1e-3

lstm_model = BiLSTMTagger(len(vocab), num_labels, EMBED_DIM, HIDDEN_DIM).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=-100)
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=LEARNING_RATE)


def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    for input_ids, labels in tqdm(loader):
        input_ids, labels = input_ids.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(input_ids)
        loss = criterion(logits.view(-1, num_labels), labels.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for input_ids, labels in loader:
            input_ids, labels = input_ids.to(device), labels.to(device)
            logits = model(input_ids)
            preds = torch.argmax(logits, dim=-1)
            for i in range(labels.size(0)):
                true_tags = [id2label[t.item()] for t in labels[i] if t.item() != -100]
                pred_tags = [
                    id2label[p.item()]
                    for p, t in zip(preds[i], labels[i])
                    if t.item() != -100
                ]
                y_true.append(true_tags)
                y_pred.append(pred_tags)
    return seq_f1_score(y_true, y_pred)


print("Training BiLSTM Tagger...")
for epoch in range(EPOCHS):
    loss = train_epoch(lstm_model, train_loader, optimizer)
    f1 = evaluate(lstm_model, valid_loader)
    print(f"Epoch {epoch+1}: Loss = {loss:.4f}, Valid F1 = {f1:.4f}")

test_f1 = evaluate(lstm_model, test_loader)
print(f"\nTest F1 BiLSTM = {test_f1:.4f}")

Training BiLSTM Tagger...


100%|██████████| 1873/1873 [00:08<00:00, 209.49it/s]


Epoch 1: Loss = 0.4839, Valid F1 = 0.5251


100%|██████████| 1873/1873 [00:06<00:00, 271.82it/s]


Epoch 2: Loss = 0.2182, Valid F1 = 0.6305


100%|██████████| 1873/1873 [00:10<00:00, 175.54it/s]


Epoch 3: Loss = 0.1528, Valid F1 = 0.6534

Test F1 BiLSTM = 0.6723


### Transformer Tagger

In [6]:
import math


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[: x.size(0), :]
        return x


class TransformerTagger(nn.Module):
    def __init__(
        self, vocab_size, label_size, embed_dim, nhead=8, num_layers=4, dropout=0.1
    ):
        super(TransformerTagger, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_encoder = PositionalEncoding(embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=nhead, dropout=dropout
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers, enable_nested_tensor=False
        )
        self.fc = nn.Linear(embed_dim, label_size)
        self.embed_dim = embed_dim

    def forward(self, src):
        src_mask = src == 0  # [batch_size, src_len]
        src = self.embedding(src) * math.sqrt(self.embed_dim)
        src = src.transpose(0, 1)  # [src_len, batch_size, embed_dim]
        src = self.pos_encoder(src)
        output = self.transformer_encoder(src, src_key_padding_mask=src_mask)
        output = output.transpose(0, 1)  # [batch_size, src_len, embed_dim]
        return self.fc(output)


transformer_model = TransformerTagger(len(vocab), num_labels, EMBED_DIM).to(device)
optimizer_tf = torch.optim.Adam(transformer_model.parameters(), lr=LEARNING_RATE)

print("Training Transformer Tagger...")
for epoch in range(EPOCHS):
    loss = train_epoch(transformer_model, train_loader, optimizer_tf)
    f1 = evaluate(transformer_model, valid_loader)
    print(f"Epoch {epoch+1}: Loss = {loss:.4f}, Valid F1 = {f1:.4f}")

test_f1_tf = evaluate(transformer_model, test_loader)
print(f"\nTest F1 Transformer = {test_f1_tf:.4f}")

Training Transformer Tagger...


100%|██████████| 1873/1873 [00:34<00:00, 54.87it/s]


Epoch 1: Loss = 0.5806, Valid F1 = 0.4090


100%|██████████| 1873/1873 [00:33<00:00, 55.25it/s]


Epoch 2: Loss = 0.3061, Valid F1 = 0.4837


100%|██████████| 1873/1873 [00:36<00:00, 50.66it/s]


Epoch 3: Loss = 0.2476, Valid F1 = 0.5042

Test F1 Transformer = 0.4920


### DistilBERT Tagger

In [10]:
from transformers import DistilBertTokenizerFast, DistilBertForTokenClassification

model_name = "distilbert-base-cased"
tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)


class BertNERDataset(Dataset):
    def __init__(self, data, tokenizer, max_len):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        tokens = item["tokens"]
        tags = [int(t) for t in item["tags"]]

        encoding = self.tokenizer(
            tokens,
            is_split_into_words=True,
            return_offsets_mapping=True,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
        )

        labels = []
        for i, doc_offset in enumerate(encoding["offset_mapping"]):
            if doc_offset[0] == 0 and doc_offset[1] != 0:
                labels.append(tags[encoding.word_ids()[i]])
            else:
                labels.append(-100)

        encoding.pop("offset_mapping")
        item = {key: torch.tensor(val) for key, val in encoding.items()}
        item["labels"] = torch.tensor(labels)
        return item


bert_train_dataset = BertNERDataset(train_data, tokenizer, MAX_LEN)
bert_valid_dataset = BertNERDataset(valid_data, tokenizer, MAX_LEN)
bert_test_dataset = BertNERDataset(test_data, tokenizer, MAX_LEN)

bert_train_loader = DataLoader(bert_train_dataset, batch_size=8, shuffle=True)
bert_valid_loader = DataLoader(bert_valid_dataset, batch_size=8)
bert_test_loader = DataLoader(bert_test_dataset, batch_size=8)

bert_model = DistilBertForTokenClassification.from_pretrained(
    model_name, num_labels=num_labels
).to(device)
bert_optimizer = torch.optim.AdamW(bert_model.parameters(), lr=2e-5)


def train_bert_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    for batch in tqdm(loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        model.zero_grad()
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate_bert(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=-1)

            for i in range(labels.size(0)):
                true_tags = [id2label[t.item()] for t in labels[i] if t.item() != -100]
                pred_tags = [
                    id2label[p.item()]
                    for p, t in zip(preds[i], labels[i])
                    if t.item() != -100
                ]
                y_true.append(true_tags)
                y_pred.append(pred_tags)
    return seq_f1_score(y_true, y_pred)


print("Training DistilBERT Tagger...")
for epoch in range(EPOCHS):
    loss = train_bert_epoch(bert_model, bert_train_loader, bert_optimizer)
    f1 = evaluate_bert(bert_model, bert_valid_loader)
    print(f"Epoch {epoch+1}: Loss = {loss:.4f}, Valid F1 = {f1:.4f}")

test_f1_bert = evaluate_bert(bert_model, bert_test_loader)
print(f"\nTest F1 DistilBERT = {test_f1_bert:.4f}")

c:\Users\livea\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\livea\.cache\huggingface\hub\models--distilbert-base-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 11092.81it/s]
DistilBertForTokenClass

Training DistilBERT Tagger...


100%|██████████| 7491/7491 [07:32<00:00, 16.54it/s]


Epoch 1: Loss = 0.1170, Valid F1 = 0.8442


100%|██████████| 7491/7491 [07:34<00:00, 16.50it/s]


Epoch 2: Loss = 0.0544, Valid F1 = 0.8458


100%|██████████| 7491/7491 [07:36<00:00, 16.42it/s]


Epoch 3: Loss = 0.0370, Valid F1 = 0.8567

Test F1 DistilBERT = 0.8688


### Results

### Report

You will write a report including the following parts:
* The description of your taggers, including the architecture, the hyperparameters, etc.
* The description of your experimental settings, including the hyperparameters you have tried,
the performance of your taggers on the dev set, etc.
* A comparison table of all models on the test set (LSTM, Transformer, DistilBERT, and
any optional models).
* Discussion and analysis of the results.